# TCC - Análise Preditiva de Risco Acadêmico Escolar
## CRISP-DM: 4. Modeling


## O que esta fase responde

A fase de Modeling responde:

- quais modelos serão testados;
- quais colunas entram como entrada dos candidatos;
- qual é o alvo de regressão;
- qual é o alvo de classificação;
- como treino, validação e teste são separados;
- qual critério escolhe o melhor modelo;
- como o modelo final é treinado depois da escolha.

Neste notebook, o contrato de dados de entrada, dados de saída e targets já vem pronto da fase anterior. Aqui o foco passa a ser comparar candidatos sobre esse contrato.



## Tarefas de aprendizagem do projeto

<div style="font-size: 1em; overflow-x: auto;">

| Tarefa | Alvo | Tipo | Métrica de escolha |
|---|---|---|---|
| Previsão de nota | `target_nota_proxima` | Regressão | menor MAE |
| Alerta de risco | `target_risco_proxima` | Classificação binária | maior F1 |

</div>

A regressão estima a próxima nota. A classificação estima se a próxima nota ficará abaixo de 6.0. As duas tarefas são treinadas na mesma função porque compartilham a mesma base temporal preparada, mas possuem alvos, candidatos e métricas diferentes.

Isso significa que o projeto não faz primeiro a regressão para depois derivar a classificação. As duas trilhas são avaliadas em paralelo sobre o mesmo recorte temporal da base.



## Carregar bibliotecas e localizar o projeto

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "school_predictor").exists() and (candidate / "artifacts").exists():
            return candidate.resolve()
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## Importar funções reais da pipeline

A demonstração usa as mesmas funções do projeto para evitar divergência entre notebook e código de produção acadêmica.





In [ ]:
from school_predictor.pipeline.config import PipelinePaths
from school_predictor.pipeline.data import load_source_tables
from school_predictor.pipeline.dataset import build_prediction_dataset, select_model_columns
from school_predictor.pipeline.modeling import (
    build_classification_baselines,
    build_classification_candidates,
    build_regression_baselines,
    build_regression_candidates,
    temporal_split,
    train_and_evaluate,
)
from school_predictor.pipeline.orchestration import resolve_mode_settings, run_real_pipeline

## Definir modo de demonstração

O projeto possui dois modos operacionais:

- `previsao_nota`: usa histórico mínimo padrão 1.
- `alerta_risco`: usa histórico mínimo padrão 2.

Mesmo quando um modo é chamado separadamente, a função de treinamento calcula regressão e classificação para manter os dois sinais comparáveis no relatório técnico.

Na prática, o `mode` altera principalmente o `min_history` padrão e o recorte da base usada no experimento. A etapa central de modelagem continua produzindo os dois sinais: nota prevista e risco previsto.





### Exemplo fixo da diferença entre os modos

A tabela abaixo resume como cada modo muda o volume de dados e quem venceu na validação:

<div style="font-size: 1em; overflow-x: auto;">

| modo | min_history | linhas_modelagem | anos_disponiveis | vencedor_regressao | vencedor_classificacao |
| --- | --- | --- | --- | --- | --- |
| previsao_nota | 1 | 106721 | 2020-2025 | RandomForest | Baseline media_duas_ultimas |
| alerta_risco | 2 | 50065 | 2021-2025 | HistGradientBoosting | HistGradientBoosting |

</div>


In [ ]:
MODE = "previsao_nota"  # alternativas: "previsao_nota" ou "alerta_risco"
MODE, MIN_HISTORY = resolve_mode_settings(MODE, min_history=None)

MODE, MIN_HISTORY

### O que este modo altera na prática

A saída desta célula mostra qual contexto operacional está sendo simulado. Quando o modo muda, o principal efeito visível é o corte de `min_history`, que altera quantas linhas entram no experimento daquele modo.

O ponto importante é que o treinamento central continua avaliando os dois sinais em paralelo: nota prevista e risco previsto.





## Carregar a base preparada para modeling

Se o dataset da pipeline já existir em `artifacts/pipeline/<modo>/student_prediction_dataset.csv`, ele pode ser carregado diretamente. Caso contrário, ele é reconstruído a partir das funções de preparação já vistas no notebook anterior.

Como o `mode` afeta o `min_history`, os recortes de `previsao_nota` e `alerta_risco` podem ter volumes diferentes mesmo vindo da mesma lógica de preparação.




In [ ]:
paths = PipelinePaths.from_project_root(PROJECT_ROOT, min_history=MIN_HISTORY, mode=MODE)

if paths.dataset_csv.exists():
    dataset = pd.read_csv(paths.dataset_csv, encoding="utf-8", low_memory=False)
else:
    source_tables = load_source_tables(paths)
    dataset = build_prediction_dataset(source_tables, min_history=MIN_HISTORY)

pd.DataFrame([{
    "modo": MODE,
    "min_history": MIN_HISTORY,
    "linhas": len(dataset),
    "colunas": len(dataset.columns),
    "alunos": dataset["IDAluno"].nunique() if "IDAluno" in dataset.columns else None,
    "disciplinas": dataset["NomeDisciplina"].nunique() if "NomeDisciplina" in dataset.columns else None,
}])

### O que a base preparada representa aqui

Ao executar a célula acima, você passa a enxergar o tamanho real da base que chegou ao experimento para este modo. Isso permite comparar como a exigência de histórico mínimo altera a quantidade de alunos, disciplinas e observações disponíveis.

Didaticamente, esta é a ponte entre Data Preparation e Modeling: a pergunta agora deixa de ser como o dado foi montado e passa a ser se ele é suficiente para comparar candidatos competitivos.





## Separação temporal: treino, validação e teste

A separação não é aleatória. O projeto usa anos letivos:

- anos mais antigos para treino;
- penúltimo ano para validação;
- último ano para teste.

Isso evita que informações do futuro vazem para o treinamento e deixa a avaliação mais parecida com o uso real na escola.

Quando existem apenas dois anos letivos disponíveis, o código usa um fallback: o último ano passa a servir ao mesmo tempo como validação e teste, porque não há um terceiro ano para separar melhor os papéis. Quando há três ou mais anos, o comportamento ideal é treino em anos mais antigos, validação no penúltimo e teste no último.





### Exemplo fixo do split temporal

Aqui fica visível que a divisão respeita o tempo, separando passado para treino e futuro para validação/teste:

<div style="font-size: 1em; overflow-x: auto;">

| modo | anos_treino | linhas_treino | ano_validacao | linhas_validacao | ano_teste | linhas_teste |
| --- | --- | --- | --- | --- | --- | --- |
| previsao_nota | 2020, 2021, 2022, 2023 | 52759 | 2024 | 27435 | 2025 | 26527 |
| alerta_risco | 2021, 2022, 2023 | 23134 | 2024 | 13688 | 2025 | 13243 |

</div>


In [ ]:
split = temporal_split(dataset)

pd.DataFrame([
    {"conjunto": "treino", "anos": split["train_years"], "linhas": len(split["train"])},
    {"conjunto": "validacao", "anos": split["validation_years"], "linhas": len(split["validation"])},
    {"conjunto": "teste", "anos": split["test_years"], "linhas": len(split["test"])},
])

### O que aconteceu com os dados após o split temporal

A base foi quebrada em passado, validação e teste sem sorteio aleatório. Isso faz com que cada conjunto tenha um papel claro:
- treino para ajustar os candidatos;
- validação para escolher o melhor;
- teste para medir o desempenho final em um ano ainda não visto.

Quando esta etapa termina, o dado já não é mais um bloco único. Ele vira um experimento temporal controlado.




## Selecionar entradas dos candidatos

Identificadores diretos e alvos não entram como features. Os candidatos recebem histórico, contexto acadêmico, faltas, pagamentos agregados, professor e sinais derivados.

Na execução real, a função `train_and_evaluate` ainda remove colunas totalmente vazias ou sem variabilidade no conjunto de treino antes de ajustar os candidatos. Isso evita manter features sem sinal útil.


### Exemplo fixo da entrada dos candidatos

Esta é uma pequena amostra da linha que os candidatos recebem para aprender e prever a próxima nota ou o risco. Os cabeçalhos estão abreviados apenas para melhorar a leitura na tela:

<div style="font-size: 1.1em; overflow-x: auto;">

| Ano | Série | Disciplina | Etapa | Nota | Nota ant. | Faltas | Pend. | Prof. disp. | Próx. nota | Risco |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | 1ª Série | Arte | 2º BIM. | 8.8 | 10.0 | 1 | 0 | 1 | 8.1 | 0 |
| 2025 | 1ª Série | Arte | 3º BIM. | 8.1 | 8.8 | 3 | 0 | 1 | 10.0 | 0 |
| 2025 | 1ª Série | Biologia | 2º BIM. | 7.0 | 6.8 | 7 | 0 | 1 | 7.4 | 0 |
| 2025 | 1ª Série | Biologia | 3º BIM. | 7.4 | 7.0 | 14 | 0 | 1 | 7.0 | 0 |
| 2025 | 1ª Série | Biologia 1 | 2º BIM. | 7.0 | 8.5 | 7 | 0 | 1 | 5.6 | 1 |

</div>

In [ ]:
feature_columns, categorical_columns = select_model_columns(dataset)

features_frame = pd.DataFrame({
    "feature": feature_columns,
    "tratamento": ["categorica" if column in categorical_columns else "numerica_ou_ordinal" for column in feature_columns],
})

features_frame

### O que esta tabela mostra sobre a entrada dos candidatos

Depois de executar a célula acima, fica visível exatamente quais colunas seguem para os algoritmos. A base preparada foi reduzida a um conjunto de sinais históricos, contextuais e categóricos que podem ser codificados e aprendidos.

Isso ajuda muito na defesa, porque transforma a pergunta "os candidatos olham para o que?" em uma resposta objetiva e auditável.




## Modelos candidatos

O projeto compara dois tipos de modelos de árvore:

- Random Forest: robusto, forte como candidato supervisionado de referência, lida bem com relações não lineares e mistura de variáveis.
- HistGradientBoosting: modelo de boosting eficiente, tende a capturar padrões tabulares mais finos quando há dados suficientes.

Também entram baselines simples baseados nas notas anteriores. Isso evita afirmar que o modelo é útil se ele não superar regras simples.

Na implementação atual, `random_forest_regressor` e `random_forest_classifier` usam `OneHotEncoder`, enquanto `hist_gradient_boosting_regressor` e `hist_gradient_boosting_classifier` usam `OrdinalEncoder`. Em ambos os casos, variáveis categóricas passam por imputação do valor mais frequente e variáveis numéricas passam por imputação pela mediana. A escolha final não é fixa: qualquer candidato, inclusive baseline, pode vencer na validação.





### Exemplo fixo dos candidatos comparados

Abaixo está um resumo simples do que é comparado nesta fase:

<div style="font-size: 1em; overflow-x: auto;">

| família | candidato | codificacao_categorica | criterio_de_escolha |
| --- | --- | --- | --- |
| Regressão | RandomForestRegressor | OneHotEncoder | menor MAE |
| Regressão | HistGradientBoostingRegressor | OrdinalEncoder | menor MAE |
| Classificação | RandomForestClassifier | OneHotEncoder | maior F1 |
| Classificação | HistGradientBoostingClassifier | OrdinalEncoder | maior F1 |
| Baseline | ultima_nota / medias_historicas | não se aplica | comparação honesta |

</div>



In [ ]:
regression_candidates = build_regression_candidates(feature_columns, categorical_columns)
classification_candidates = build_classification_candidates(feature_columns, categorical_columns)

pd.DataFrame([
    {"familia": "regressao", "candidato": name}
    for name in regression_candidates.keys()
] + [
    {"familia": "classificacao", "candidato": name}
    for name in classification_candidates.keys()
] + [
    {"familia": "baseline_regressao", "candidato": name}
    for name in build_regression_baselines(dataset).keys()
] + [
    {"familia": "baseline_classificacao", "candidato": name}
    for name in build_classification_baselines(dataset).keys()
])

### O que os candidatos significam quando comparados lado a lado

A saída acima mostra que a modelagem não parte do pressuposto de que um algoritmo complexo sempre será melhor. Baselines simples entram na disputa com os modelos supervisionados. No caso da classificação, essas referências simples nascem das previsões baseline de nota convertidas em regra de risco com corte em `6.0`.

Na prática, isso quer dizer que o projeto só considera ganho real quando um candidato supera regras que já seriam razoáveis sem aprendizado de máquina.



## Por que não usar apenas ifs?

Regras como `se última nota < 6, então risco` são baselines importantes, mas são limitadas. Elas olham para um corte fixo e ignoram combinações como queda de tendência, histórico anterior, faltas, média da coorte, pagamentos agregados, série, etapa e disciplina.

A classificação aprende padrões combinados. Um aluno com nota atual acima de 6 pode ainda receber alerta se o histórico indicar queda, acumulado de faltas ou comportamento parecido com alunos que caíram abaixo da linha na etapa seguinte. Da mesma forma, um aluno com nota baixa pode não ser classificado como risco alto se a tendência e o contexto indicarem recuperação.





## Critérios de escolha

- Regressão: o melhor candidato é o que tiver menor MAE na validação.
- Classificação: o melhor candidato é o que tiver maior F1 na validação.

O MAE mede, em média, quantos pontos a previsão de nota erra. O F1 equilibra precision e recall, evitando olhar apenas para acerto geral quando o evento de risco pode ser menor que a quantidade de alunos sem risco. Outras métricas, como RMSE, acerto dentro de meio ponto, precision e recall, continuam sendo calculadas para apoiar a leitura posterior, mas não decidem o vencedor principal nesta fase.




### Exemplo fixo dos critérios e vencedores

A escolha final depende da métrica principal de cada tarefa e pode até eleger um baseline:

<div style="font-size: 1em; overflow-x: auto;">

| modo | alvo | fonte_vencedora | vencedor | metrica_usada | valor_na_validacao |
| --- | --- | --- | --- | --- | --- |
| previsao_nota | regressão | modelo | RandomForest | MAE | 0.8152 |
| previsao_nota | classificação | baseline | media_duas_ultimas | F1 | 0.7359 |
| alerta_risco | regressão | modelo | HistGradientBoosting | MAE | 0.8124 |
| alerta_risco | classificação | modelo | HistGradientBoosting | F1 | 0.7564 |

</div>


## Executar treinamento sob demanda

Por padrão, o treinamento fica desligado para manter o notebook leve e seguro para GitHub. Para rodar localmente, altere `RUN_TRAINING = True`.

A função `train_and_evaluate` executa a seleção dos candidatos, compara modelos e baselines na validação, re-treina os vencedores com treino + validação e retorna métricas e saídas de apoio para leitura técnica. No teste final, a comparação de referência usa o baseline `ultima_nota`. Ela não salva arquivos. Para salvar artefatos iguais aos da CLI, use `run_real_pipeline` no bloco opcional seguinte.


### Exemplo fixo do que a modelagem devolve

Quando a fase de modeling termina, a execução local devolve estruturas técnicas que documentam a decisão tomada nesta fase:

<div style="font-size: 1em; overflow-x: auto;">

| saida_tecnica | o_que_contem |
| --- | --- |
| metrics | ranking de candidatos na validação, split temporal e vencedores por tarefa |
| models | pipeline final re-treinada quando um modelo supervisionado vence; se um baseline vencer, essa tarefa segue sem modelo supervisionado final |
| predictions | predições do conjunto de teste com baseline de referência |
| error_analysis | tabelas auxiliares que serão exploradas com mais detalhe na fase de Evaluation |

</div>


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    training_output = train_and_evaluate(dataset, feature_columns, categorical_columns)
    metrics = training_output["metrics"]

    display(pd.DataFrame([
        {"grupo": "regressao_modelo", **metrics["regression_model"]},
        {"grupo": "regressao_baseline", **metrics["regression_baseline"]},
    ]))
    display(pd.DataFrame([
        {"grupo": "classificacao_modelo", **metrics["classification_model"]},
        {"grupo": "classificacao_baseline", **metrics["classification_baseline"]},
    ]))
    display(pd.DataFrame([
        {"tipo": "regressao", **metrics["selected_regression_candidate"]},
        {"tipo": "classificacao", **metrics["selected_classification_candidate"]},
    ]))
else:
    print("Treinamento nao executado. Altere RUN_TRAINING para True para rodar localmente.")

### O que acontece quando o treinamento roda

Se você ativar `RUN_TRAINING`, a pipeline vai testar todos os candidatos na validação, escolher o melhor por métrica da tarefa e depois re-treinar esse vencedor com treino + validação.

A saída que aparece depois disso não é apenas um número final. Ela mostra:
- desempenho do candidato vencedor;
- desempenho do baseline de referência (`ultima_nota`);
- quem venceu em regressão;
- quem venceu em classificação.



## Opcional: rodar a pipeline do modo e salvar artefatos

Este bloco reproduz a execução operacional da CLI para um modo específico. Ele salva dataset, modelo, predições e relatório técnico em `artifacts/pipeline/<modo>/`. Aqui ele aparece apenas como apoio opcional de rastreabilidade local, não como foco conceitual da fase.



In [ ]:
RUN_AND_SAVE_ARTIFACTS = False

if RUN_AND_SAVE_ARTIFACTS:
    output = run_real_pipeline(project_root=PROJECT_ROOT, mode=MODE, min_history=MIN_HISTORY)
    print(output["paths"].output_dir)

## Resultado da fase

Ao final da modelagem, o projeto possui:

- candidato escolhido para regressão, por menor MAE na validação;
- candidato escolhido para classificação, por maior F1 na validação;
- comparação com baselines simples;
- re-treinamento usando treino + validação;
- teste final no ano mais recente;
- registro técnico de quais candidatos venceram e com quais métricas.

Esses resultados seguem para a fase de Evaluation, onde as métricas serão interpretadas e os erros serão analisados em detalhe.

O uso da palavra candidato é importante aqui porque o vencedor pode ser um modelo de árvore ou um baseline simples, dependendo do desempenho observado na validação.



## Leitura didática da fase

No fim deste notebook, o dado já passou por uma mudança conceitual importante:
- antes, ele era apenas uma base temporal preparada;
- agora, ele virou experimento comparativo entre candidatos.

A fase de Modeling, portanto, não entrega apenas um modelo salvo. Ela entrega uma decisão justificável sobre qual candidato merece seguir como referência técnica para o teste final.

